# Ray


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from starwinds_readplt.dataset import Dataset

from batcamp import Octree
from batcamp import OctreeInterpolator
from batcamp import OctreeRayInterpolator
from batcamp import OctreeRayTracer

from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from batcamp.ray import HEX_TETS_INDEX, TET_FACES_INDEX

import pooch


### Pick spherical sample file


In [ ]:
path = pooch.retrieve(
    url='https://zenodo.org/records/7110555/files/run-Sun-G2211.tar.gz',
    known_hash='c31a32aab08cc20d5b643bba734fd7220e6b369e691f55f88a3a08cc5b2a2136',
    progressbar=False,
    processor=pooch.Untar(members=['run-Sun-G2211/SC/IO2/3d__var_4_n00044000.plt']),
)[0]
path

### Build dataset, tree, and interpolator


In [ ]:
ds = Dataset.from_file(path)
print(ds)

interp = OctreeInterpolator(ds, ['Rho [g/cm^3]'])
print(interp)


### Ray walk and sampled values along the ray


In [ ]:
r_shell = 4.0
p_start = r_shell * np.array([1.0, 0.0, 0.0], dtype=float)
p_end = r_shell * np.array([-0.25, 0.96, 0.11], dtype=float)
p_end = r_shell * (p_end / np.linalg.norm(p_end))
origin = p_start
direction = p_end - p_start
t0, t1 = 0.0, float(np.linalg.norm(direction))
segments = OctreeRayTracer(interp.tree).trace(origin, direction, t0, t1)
print('ray start:', p_start, 'r=', float(np.linalg.norm(p_start)))
print('ray end:', p_end, 'r=', float(np.linalg.norm(p_end)))
print('segment count:', len(segments))
if segments:
    print('first segment:', segments[0])
    print('last segment:', segments[-1])
n_samples = 1200
t_values, ray_values, ray_cell_ids, _ = OctreeRayInterpolator(interp).sample(origin, direction, t0, t1, n_samples)
print('sample count:', n_samples)
print('nan fraction:', float(np.mean(~np.isfinite(ray_values))))


### Piecewise linear functions over the ray via tet split


In [ ]:
pieces = OctreeRayInterpolator(interp).linear_pieces(origin, direction, t0, t1)
print('linear piece count:', len(pieces))
if pieces:
    print('coverage:', pieces[0].t_start, '->', pieces[-1].t_end)
    for i, p in enumerate(pieces[:6]):
        print(i, (p.t_start, p.t_end), 'cell', p.cell_id, 'tet', p.tet_id, 'slope', float(np.asarray(p.slope)), 'intercept', float(np.asarray(p.intercept)))


### Evaluate the piecewise linear model on the sample t-grid and compare with direct interpolation.


In [ ]:
piece_vals = np.full_like(ray_values, np.nan)
idx = 0
for i, t in enumerate(t_values):
    while idx + 1 < len(pieces) and t > pieces[idx].t_end:
        idx += 1
    if idx < len(pieces):
        p = pieces[idx]
        if p.t_start - 1e-8 <= t <= p.t_end + 1e-8:
            piece_vals[i] = p.slope * t + p.intercept
mask = np.isfinite(piece_vals) & np.isfinite(ray_values)
print('piece finite fraction:', float(np.mean(np.isfinite(piece_vals))))
if np.any(mask):
    diff = np.abs(piece_vals[mask] - ray_values[mask])
    print('max abs diff:', float(np.max(diff)))
    print('median abs diff:', float(np.median(diff)))


### Exact integration of piecewise linear model:

integral_{a}^{b} (m*t + b0) dt = 0.5*m*(b^2-a^2) + b0*(b-a)


In [ ]:
integral_exact = np.asarray(0.0, dtype=float)
for p in pieces:
    a = float(p.t_start)
    b = float(p.t_end)
    integral_exact = integral_exact + 0.5 * p.slope * (b * b - a * a) + p.intercept * (b - a)
# Numerical check against dense trapezoid on ray samples
integral_trap = np.trapezoid(ray_values, t_values)
print('exact piecewise-linear integral:', float(np.asarray(integral_exact)))
print('trapz(ray samples):', float(np.asarray(integral_trap)))
print('abs difference:', float(np.abs(np.asarray(integral_exact) - np.asarray(integral_trap))))


### 3D ray traversal with per-cell colors and hexahedral (cube) cell rendering.


In [ ]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
if not segments:
    raise RuntimeError('Ray did not traverse any octree cells; adjust ray setup.')
ray_dir = direction / np.linalg.norm(direction)
n_seg = len(segments)
cmap = plt.colormaps['turbo']
colors = cmap(np.linspace(0.0, 1.0, max(n_seg, 2)))
def cube_index(r_bit, polar_bit, azimuth_bit):
    return int(r_bit + 2 * polar_bit + 4 * azimuth_bit)
cube_faces = [
    [cube_index(0, 0, 0), cube_index(0, 1, 0), cube_index(0, 1, 1), cube_index(0, 0, 1)],
    [cube_index(1, 0, 0), cube_index(1, 0, 1), cube_index(1, 1, 1), cube_index(1, 1, 0)],
    [cube_index(0, 0, 0), cube_index(0, 0, 1), cube_index(1, 0, 1), cube_index(1, 0, 0)],
    [cube_index(0, 1, 0), cube_index(1, 1, 0), cube_index(1, 1, 1), cube_index(0, 1, 1)],
    [cube_index(0, 0, 0), cube_index(1, 0, 0), cube_index(1, 1, 0), cube_index(0, 1, 0)],
    [cube_index(0, 0, 1), cube_index(0, 1, 1), cube_index(1, 1, 1), cube_index(1, 0, 1)],
]
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
all_faces = []
all_facecolors = []
extent_points = []
for idx, seg in enumerate(segments):
    color = colors[idx]
    cid = int(seg.cell_id)
    p0 = origin + float(seg.t_enter) * ray_dir
    p1 = origin + float(seg.t_exit) * ray_dir
    pm = 0.5 * (p0 + p1)
    ax.plot([p0[0], p1[0]], [p0[1], p1[1]], [p0[2], p1[2]], color=color, linewidth=3.0)
    ax.text(pm[0], pm[1], pm[2], str(idx), color=color, fontsize=7)
    corner_ids = interp._corners[cid]
    cell_xyz_raw = interp.lookup.points[corner_ids]
    corner_order = interp._bin_to_corner[cid]
    cell_xyz = cell_xyz_raw[corner_order]
    for face in cube_faces:
        all_faces.append(cell_xyz[np.array(face, dtype=np.int64)])
        all_facecolors.append(color)
    extent_points.extend([p0, p1])
    extent_points.extend(cell_xyz)
poly = Poly3DCollection(
    all_faces,
    facecolors=all_facecolors,
    # edgecolors=all_facecolors,
    linewidths=0.4,
    alpha=0.12,
)
ax.add_collection3d(poly)
extent = np.array(extent_points, dtype=float)
mins = np.min(extent, axis=0)
maxs = np.max(extent, axis=0)
center = 0.5 * (mins + maxs)
radius = 0.5 * float(np.max(maxs - mins))
ax.set_xlim(center[0] - radius, center[0] + radius)
ax.set_ylim(center[1] - radius, center[1] + radius)
ax.set_zlim(center[2] - radius, center[2] + radius)
ax.set_xlabel('X [R]')
ax.set_ylabel('Y [R]')
ax.set_zlabel('Z [R]')
ax.set_title('Ray traversal with hexahedral cell surfaces (r=4 to r=4 endpoints)')
norm = plt.Normalize(vmin=0, vmax=max(n_seg - 1, 1))
sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
cbar.set_label('ray segment index (0 = first cell)')
plt.tight_layout()
plt.show()
